Pasos para armar modelo

# 1. Levantar la información y definir el problema
- Entrevista con stakeholders
- Estudiar el contexto del problema
- Definir métricas claves del modelo

# 2. Recolección de datos
- Acceder a las bases de datos encesarias 
- Integrar datos de múltiples fuentes si es necesario

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
# Librerías necesarias para análisis de datos, visualización y preprocesamiento
import pandas as pd  # Manejo de datos tabulares
import numpy as np  # Operaciones matemáticas y matrices
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, RobustScaler  # Codificación y escalado
from sklearn.impute import SimpleImputer, KNNImputer  # Imputación de valores nulos
import seaborn as sns  # Visualización de datos
import matplotlib.pyplot as plt  # Gráficos

# 3. Exploratory Data Analysis
- Análisis descriptivo
- Visualización
- etectar valores atípicos y datos faltantes (nulos, outliers, duplicados, incnsistencias)

## 3.1 Comprensión inicial de datos

In [ ]:
# Miro un poco la información
df = pd.read_csv("books.csv")
df.head()

In [ ]:
# Obtengo información de cuántos nulos hay y el tipo de dato
df.info()

In [ ]:
# Estadística descriptiva
print(df.describe())

In [ ]:
# Me fijo el tipo de dato de cada uno, tengo que validar el tipo de dato tenga sentido. 
df.dtypes

In [ ]:
# Verifico la cantida de filas y columnas
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

In [ ]:
# Me fijo en las varaibles categóricas qué opciones hay y cuáles hay de cada una
print(df['variable_categorica'].value_counts())

## 3.2 Análisis de valores nulos

In [ ]:
# Me fijo si hay nulos
print(df.isna().sum())
print(df.isnull().sum())

Eliminación de filas/columnas: Si el porcentaje de nulos es alto (e.g., > 40%).

Imputación:

Numéricas: Media, mediana, moda o técnicas avanzadas como KNN imputer (from sklearn.impute import KNNImputer).
Categóricas: Rellenar con la moda o una nueva categoría ("Desconocido")

In [ ]:
# Visualizo los valores nulos
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.show()

## 3.3 Análisis de outliers

In [ ]:
# Reglas de 1.5*IQR
Q1 = df['variable'].quantile(0.25)
Q3 = df['variable'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['variable'] < Q1 - 1.5 * IQR) | (df['variable'] > Q3 + 1.5 * IQR)]
print(outliers)

In [ ]:
# Z-score
from scipy.stats import zscore
df['zscore'] = zscore(df['variable'])
print(df[df['zscore'].abs() > 3])

Tratamiento de outliers:

- Eliminación: Si son errores claros o irrelevantes.
- Winsorización: Reemplazar valores extremos por percentiles (e.g., 5% y 95%).
- Transformaciones: Aplicar logaritmo, raíz cuadrada o escalado robusto (RobustScaler de scikit-learn).

In [ ]:
# Visualización de outliers
sns.boxplot(x=df['variable'])
plt.show()

## 3.4 Análisis de correlación

In [ ]:
# Matriz de correlación
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.show()

Identificación de relaciones fuertes:

Revisa valores de correlación altos (>0.8) o bajos (<-0.8) para eliminar variables redundantes (multicolinealidad).
Si usas variables categóricas, emplea Cramer’s V para analizar relaciones.

In [ ]:
# Pairplot
sns.pairplot(df)
plt.show()

In [ ]:
# Scatter plot
sns.scatterplot(x='variable1', y='variable2', data=df)

## 3.5 Análisis de variables categóricas

In [ ]:
# Distribución de categorías
df['variable_categorica'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
# Transformación de variables categóricas
# Label Encoding (si hay odinalidad)
from sklearn.preprocessing import LabelEncoder
df['variable_categorica'] = LabelEncoder().fit_transform(df['variable_categorica'])

In [ ]:
# One-Hot Encoding (para categorías nominales)
df = pd.get_dummies(df, columns=['variable_categorica'], drop_first=True)

## 3.6 Análisis de distribuciones

In [ ]:
# Variables numéricas
# Histograma
df['variable'].hist(bins=20)
plt.show()

In [ ]:
# Para visualizar variables numéricas
sns.histplot(data=df, x="varaible")
plt.show()

In [ ]:
sns.histplot(data=df, x="varaible", binwidth=.1)
plt.show()

In [ ]:
# Variables numéricas
# KDE (Kernel Density Estimation)
sns.kdeplot(df['variable'], shade=True)
plt.show()

In [ ]:
# Transformación de datos
# Escalado (para modelos sensibles como SVM o KNN)
from sklearn.preprocessing import StandardScaler
df['variable_scaled'] = StandardScaler().fit_transform(df[['variable']])

## 3.7 Interacción entre variables

In [ ]:
# Análisis bivariado
# Categórico vs. numérico
sns.boxplot(x='categoria', y='variable_numerica', data=df)

In [ ]:
# Dos categorías
sns.scatterplot(x='variable1', y='variable2', data=df)

## 3.8 Otras cosas a hacer

In [ ]:
# Valores duplicados
print(df.duplicated().sum())
df = df.drop_duplicates()

# 4. Preprocesamiento de Datos

Objetivos:
    Preparar los datos para que sean adecuados para el modelo.

Acciones:
- Limpieza de datos: Imputar valores nulos (por ejemplo, usar la media o modelos para estimarlos); corregir errores en los datos; eliminar duplicados.
- Codificación de variables categóricas: Usar técnicas como One-Hot Encoding o codificación ordinal.
- Escalado y normalización: Escalar datos numéricos (por ejemplo, MinMaxScaler o StandardScaler).
- Transformación: Corregir distribuciones sesgadas (aplicando logaritmos, raíces cuadradas, etc.).
- Equilibrio de clases (si aplica): Usar técnicas como SMOTE o undersampling.

## 4.1 Limpieza de datos

### 4.1.1 Valores nulos

In [ ]:
# Muestra la cantidad de valores nulos por columna
print("Valores nulos por columna:")
print(df.isnull().sum())

In [ ]:
# Representación gráfica de los valores nulos como un mapa de calor
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Mapa de calor de valores nulos")
plt.show()

In [ ]:
# Reemplaza valores nulos de variables numéricas con la media (puede usarse mediana también)
num_imputer = SimpleImputer(strategy='mean')
df['variable_numerica'] = num_imputer.fit_transform(df[['variable_numerica']])

In [ ]:
# Reemplaza valores nulos de variables categóricas con el valor más frecuente
cat_imputer = SimpleImputer(strategy='most_frequent')
df['variable_categorica'] = cat_imputer.fit_transform(df[['variable_categorica']])

In [ ]:
# Usa el algoritmo KNN para estimar y reemplazar valores nulos basándose en vecinos cercanos
knn_imputer = KNNImputer(n_neighbors=5)
df[['variable1', 'variable2']] = knn_imputer.fit_transform(df[['variable1', 'variable2']])

### 4.1.2 Detección y tratamiento de outliers

In [ ]:
# Calcula los límites inferior y superior para detectar outliers usando la Regla de 1.5 * IQR
Q1 = df['variable_numerica'].quantile(0.25)  # Primer cuartil
Q3 = df['variable_numerica'].quantile(0.75)  # Tercer cuartil
IQR = Q3 - Q1  # Rango intercuartil
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identifica las filas que contienen valores fuera de los límites
outliers = df[(df['variable_numerica'] < lower_bound) | (df['variable_numerica'] > upper_bound)]
print(f"Cantidad de outliers: {len(outliers)}")

In [ ]:
# Gráfico tipo boxplot para visualizar los valores atípicos
sns.boxplot(x=df['variable_numerica'])
plt.title("Detección de outliers")
plt.show()

In [ ]:
# Tratamiento de outliers
# Winsorización
# Limita los valores extremos al rango permitido (no elimina valores, solo los ajusta)
df['variable_numerica'] = np.clip(df['variable_numerica'], lower_bound, upper_bound)

In [ ]:
# Tratameinto de outliers
# Elimino valores extemos
# Filtra el dataframe para eliminar filas con valores atípicos
df = df[(df['variable_numerica'] >= lower_bound) & (df['variable_numerica'] <= upper_bound)]

## 4.2 Codificación de variabls categóricas

In [ ]:
# Labl Encoding (variables ordinales)
# Convierte categorías en números enteros (útil para variables categóricas ordinales)
label_encoder = LabelEncoder()
df['variable_categorica'] = label_encoder.fit_transform(df['variable_categorica'])

In [ ]:
# One-Hot Encoding(variables nominales)
# Crea nuevas columnas binarias para cada categoría (útil para variables categóricas nominales)
df = pd.get_dummies(df, columns=['variable_categorica'], drop_first=True)

## 4.3 Escalado de datos

In [ ]:
# Escalado estándar (algritmos como SVM o PCA)
# Normaliza los valores para que tengan media 0 y desviación estándar 1
scaler = StandardScaler()
df['variable_numerica_escalada'] = scaler.fit_transform(df[['variable_numerica']])

In [ ]:
# Escalado robusto (para datos con outliers)
# Escala los valores considerando la mediana y el rango intercuartil (menos sensible a outliers)
robust_scaler = RobustScaler()
df['variable_numerica_robusta'] = robust_scaler.fit_transform(df[['variable_numerica']])

# 5. Generación de nuevas características

In [ ]:
# Variables polinómicas
# Crea una nueva columna con el valor al cuadrado y otra con el logaritmo natural
df['variable_cuadrada'] = df['variable_numerica'] ** 2
df['variable_log'] = np.log1p(df['variable_numerica'])  # Logaritmo con ajuste para evitar log(0)

In [ ]:
# Razones entre variables
# Crea una nueva columna como la relación entre dos variables (evitando división por 0)
df['razon_variable'] = df['variable1'] / (df['variable2'] + 1e-10)

In [ ]:
# Binning (agrupacion de valores continuos en categorias)
# Agrupa valores continuos en categorías según rangos definidos
df['rango_variable'] = pd.cut(df['variable_numerica'], bins=3, labels=['Bajo', 'Medio', 'Alto'])

# 6. División de Datos

- Objetivos: Separar los datos en conjuntos de entrenamiento, validación y prueba.
- Acciones: Proporciones comunes: 70% entrenamiento, 15% validación, 15% prueba.
- Estrategias avanzadas:
    - División estratificada (para mantener proporciones de clases).
    - Validación cruzada (k-fold).

In [8]:
# Importamos las librerías necesarias para dividir los datos
from sklearn.model_selection import train_test_split

## 6.1 Cargar y preparar los datos
Asegúrate de tener tus datos listos para ser divididos, con variables dependientes e independientes definidas. Aquí tomamos como ejemplo que df es tu DataFrame y target es la columna objetivo (variable dependiente).

In [ ]:
# Definir las variables independientes (X) y la variable dependiente (y)
X = df.drop('target', axis=1)  # Eliminar la columna objetivo
y = df['target']  # Solo la columna objetivo

## 6.2 División de datos en entrenamiento, validación y prueba

### 6.2.1 División de datos en entrenamiento y prueba
Aquí se divide en entrenamiento (70%) y prueba (30%), de modo que se tiene un conjunto de datos para entrenar el modelo y otro para evaluarlo.

In [ ]:
# Dividir los datos en un conjunto de entrenamiento (70%) y un conjunto de prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

¿Por qué 70% entrenamiento y 30% prueba?
Un 70% de datos para entrenamiento suele ser suficiente para entrenar el modelo sin sobreajustarlo.
Un 30% para la prueba permite tener una muestra suficientemente grande para evaluar la precisión del modelo.
En algunos casos, se usa 80% para entrenamiento y 20% para prueba, pero este ajuste depende de cuántos datos tengamos y de lo que busquemos en la evaluación.

### 6.2.2 División de los datos de entrenamiento en entrenamiento y validación
Posteriormente, el conjunto de entrenamiento (70%) se puede dividir en entrenamiento (85%) y validación (15%), de modo que podamos ajustar el modelo y validar su rendimiento durante el proceso de entrenamiento.

In [ ]:
# Dividir el conjunto de entrenamiento (70%) en subconjuntos de entrenamiento (85%) y validación (15%)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

¿Por qué 15% para validación?
Usar un 15% de los datos para validación es una opción común. Este conjunto de validación permite ajustar los parámetros del modelo sin comprometer la capacidad de generalización del modelo (sin usar los datos de prueba).
Si tenemos pocos datos, podríamos optar por 10% para validación, pero en general un 15% proporciona un buen equilibrio para evaluar el rendimiento sin sobreajustar.

Resumen de las proporciones
- 70% para entrenamiento: Permite que el modelo aprenda de una buena cantidad de datos.
- 15% para validación: Permite ajustar hiperparámetros durante el entrenamiento.
- 15% para prueba: Permite evaluar el rendimiento del modelo de forma objetiva y asegurarse de que no haya sobreajuste.

In [ ]:
# Verificar el tamaño de cada conjunto de datos
print(f"Tamaño de datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Tamaño de datos de validación: {X_val.shape[0]} filas")
print(f"Tamaño de datos de prueba: {X_test.shape[0]} filas")

## 6.3 Alternativa con validación cruzada (K-fold cross-validation)
Si prefieres no tener un conjunto de validación fijo, puedes usar validación cruzada (K-fold cross-validation) para evaluar el modelo durante el entrenamiento. Esta es una buena opción si tienes un conjunto de datos pequeño.

In [ ]:
from sklearn.model_selection import cross_val_score

# Realizar validación cruzada con 5 pliegues
scores = cross_val_score(model, X_train, y_train, cv=5)
print(f"Scores de validación cruzada: {scores}")
print(f"Precisión promedio: {scores.mean()}")

¿Por qué usar validación cruzada?
La validación cruzada utiliza diferentes subconjuntos del conjunto de entrenamiento para entrenar y validar el modelo varias veces. Esto da una estimación más robusta de cómo el modelo podría comportarse en datos nuevos, especialmente si tenemos pocos datos.

Conclusión

La división de datos en entrenamiento, validación y prueba es crucial para evitar el sobreajuste y garantizar que el modelo generalice correctamente a nuevos datos. Los pasos comunes son:

Dividir en entrenamiento (70%) y prueba (30%).
Dividir el conjunto de entrenamiento (70%) en entrenamiento (85%) y validación (15%).
Usar validación cruzada si tienes pocos datos.
Este proceso ayuda a tener datos suficientes para entrenar, ajustar y evaluar el modelo de forma adecuada, manteniendo una buena capacidad de generalización.

# 7. Selección del Modelo
- Objetivos: Elegir un modelo adecuado para el problema y los datos.
- Determinar el tipo de modelo:
    - Regresión (si la salida es numérica).
    - Clasificación (si la salida es categórica).
    - Clustering o reducción de dimensionalidad (si es no supervisado).
- Explorar múltiples modelos:
    - Regresión: Regresión Lineal, Gradient Boosting.
    - Clasificación: Random Forest, SVM, XGBoost.
    - Clustering: k-Means, DBSCAN.

## 7.1 Regresión Lineal (Supervisado - Regresión)
Este modelo se usa cuando la variable dependiente es continua.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Datos de ejemplo
X = df.drop('target', axis=1)  # Variables independientes
y = df['target']  # Variable dependiente

# División en entrenamiento y prueba (70% entrenamiento, 30% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Modelo de regresión lineal
model = LinearRegression()
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

# Evaluación del modelo
mse = mean_squared_error(y_test, y_pred)
print(f"Error cuadrático medio (MSE): {mse}")

Explicación:

El modelo LinearRegression() ajusta una línea recta a los datos de entrenamiento.
Usamos el MSE para evaluar la precisión del modelo.

## 7.2 Regresión con Random Forest (Supervisado - Regresión)
Este modelo es útil cuando las relaciones no son lineales.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Modelo de Random Forest para regresión
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf_model.predict(X_test)

# Evaluación del modelo
mse_rf = mean_squared_error(y_test, y_pred_rf)
print(f"Error cuadrático medio (MSE) - Random Forest: {mse_rf}")

Explicación:

RandomForestRegressor() crea un conjunto de árboles de decisión y promedia sus predicciones para mejorar la precisión y evitar el sobreajuste.

Relaciones no son lineales en el contexto de Random Forest (y en general, en machine learning), se refiere a que la relación entre las características (variables independientes) y la variable objetivo (dependiente) no sigue una estructura matemática sencilla, como una línea recta.

## 7.3 Clasificación con Regresión Logística (Supervisado - Clasificación)
Este modelo es útil para problemas de clasificación binaria.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Datos de ejemplo
X = df.drop('target', axis=1)  # Variables independientes
y = df['target']  # Variable dependiente (binaria)

# División en entrenamiento y prueba (70% entrenamiento, 30% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Modelo de regresión logística
logreg_model = LogisticRegression()
logreg_model.fit(X_train, y_train)

# Predicciones
y_pred_logreg = logreg_model.predict(X_test)

# Evaluación del modelo
accuracy = accuracy_score(y_test, y_pred_logreg)
print(f"Precisión de la Regresión Logística: {accuracy}")

Explicación:

LogisticRegression() es un modelo lineal usado para predecir probabilidades, ideal para clasificación binaria.

## 7.4 XGBoost (Supervisado - Regresión y Clasificación)
XGBoost es un potente algoritmo basado en el boosting, usado para problemas tanto de regresión como de clasificación.

### 7.4.1 Para Regresión

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Modelo de XGBoost para regresión
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
xgb_model.fit(X_train, y_train)

# Predicciones
y_pred_xgb = xgb_model.predict(X_test)

# Evaluación del modelo
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
print(f"Error cuadrático medio (MSE) - XGBoost: {mse_xgb}")

### 7.4.2 Para Clasificación

In [ ]:
from sklearn.metrics import accuracy_score

# Modelo de XGBoost para clasificación
xgb_clf_model = xgb.XGBClassifier(objective='binary:logistic', n_estimators=100)
xgb_clf_model.fit(X_train, y_train)

# Predicciones
y_pred_xgb_clf = xgb_clf_model.predict(X_test)

# Evaluación del modelo
accuracy_xgb = accuracy_score(y_test, y_pred_xgb_clf)
print(f"Precisión de XGBoost (Clasificación): {accuracy_xgb}")

Explicación:

XGBRegressor() y XGBClassifier() son las implementaciones de XGBoost para problemas de regresión y clasificación respectivamente. XGBoost es muy eficiente para manejar grandes cantidades de datos y evitar el sobreajuste.

El objetivo del boosting es combinar múltiples modelos (usualmente modelos simples, como árboles de decisión) para formar un modelo final más preciso y robusto.
Cada modelo sucesivo corrige los errores cometidos por el modelo anterior. En otras palabras, cada nuevo modelo se enfoca en mejorar las predicciones en los ejemplos que los modelos anteriores han clasificado incorrectamente.

Tipos de modeos:
- AdaBoost
- Gradient Boosting
- XGBoost
- LightGBM

## 7.5 Support Vector Machine (SVM - Supervisado - Clasificación y Regresión)
El SVM puede ser utilizado tanto para clasificación como para regresión.

### 7.5.1 Para Clasificación

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Modelo SVM para clasificación
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)

# Predicciones
y_pred_svm = svm_model.predict(X_test)

# Evaluación del modelo
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Precisión del SVM: {accuracy_svm}")

### 7.5.2 Para Regresión

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

# Modelo SVM para regresión
svr_model = SVR(kernel='linear')
svr_model.fit(X_train, y_train)

# Predicciones
y_pred_svr = svr_model.predict(X_test)

# Evaluación del modelo
mse_svr = mean_squared_error(y_test, y_pred_svr)
print(f"Error cuadrático medio (MSE) - SVM: {mse_svr}")

Explicación:

SVC() y SVR() son las versiones de SVM para clasificación y regresión respectivamente. SVM es eficaz en problemas donde las clases no son linealmente separables y cuando el número de características es alto.

Clases no son linealmente separables, se hace referencia a un escenario en el que las dos clases a predecir no se pueden separar por una línea recta (o un hiperplano en dimensiones superiores) en el espacio de características. Esto significa que no es posible encontrar un único hiperplano que divida de manera perfecta las dos clases.

Imagina un conjunto de datos donde los puntos de una clase están agrupados alrededor de un círculo y los de la otra clase están agrupados en la zona exterior del círculo. Una línea recta no puede separar las clases de manera efectiva en este caso.

- Kernel lineal:
Descripción: Este es el kernel más simple y se utiliza cuando las clases son linealmente separables, o cuando quieres que el modelo funcione en el espacio original de características sin aplicar ninguna transformación.
Uso: Se usa cuando el conjunto de datos tiene una separación clara y lineal entre las clases.

- Kernel polinómico:
Descripción: Este kernel mapea los datos a un espacio de mayor dimensión de forma polinómica, donde 
𝑑
d es el grado del polinomio, y 
𝑐
c es un parámetro de sesgo.
Uso: Es útil cuando las relaciones entre las clases pueden representarse como una combinación polinómica de las características.

- Kernel Radial Basis Function (RBF) o Gaussiano
Descripción: El kernel RBF mide la similitud entre dos puntos, tomando en cuenta la distancia euclidiana entre ellos. Este kernel transforma los datos de manera no lineal, lo que permite que el SVM cree una frontera de decisión muy flexible, adecuada para casos de clases no linealmente separables.
Uso: Es el kernel más común en situaciones donde las clases no son linealmente separables, ya que puede mapear los datos a un espacio de características de alta dimensión y crear fronteras de decisión complejas y no lineales.

- Kernel Sigmoide
Descripción: Este kernel se basa en la función sigmoide y se comporta de manera similar a una red neuronal de una sola capa. Sin embargo, a menudo no se utiliza tanto como el RBF.
Uso: Es útil en redes neuronales y en situaciones donde se busca modelar la no linealidad en las relaciones de los datos.

Si tienes motivos para creer que las relaciones entre las características y la clase no pueden separarse por una línea recta (como en el caso de datos de imagen, texto o datos con interacciones complejas), el kernel RBF es una opción común debido a su flexibilidad y su capacidad para manejar datos con patrones complejos.

## 7.6 Reducción de Dimensionalidad con PCA (No Supervisado)
La reducción de dimensionalidad es útil para manejar datos con muchas variables, manteniendo la mayor cantidad de información.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Datos de ejemplo
X = df.drop('target', axis=1)

# PCA: Reducción a 2 dimensiones
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Visualización de los datos proyectados
plt.scatter(X_pca[:, 0], X_pca[:, 1])
plt.title("Reducción de Dimensionalidad con PCA")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.show()

Explicación:

PCA() reduce el número de características, transformando los datos a un espacio de menor dimensión. Es útil cuando se tienen muchas características y se desea simplificar el modelo sin perder demasiada información.

Reducción de simensionaldiad:
- Se reduce el numeor de variables
- cuando hay muchas características uqe pued ellevar l sobreajuste
- Elimina características irrelevantes
- Mejora eficiencia computacional
- Previene el sobreajuste
- Mejora la visualzaición
- Disminuye la dimensionalidad
- Tecnicas son
    - PCA
    - t-SNE
    - Autoencoders
- Se usa cuando:
    - los datos tienen muchas caracteristicas y se desea mejorar la intepretabilidad
    cuando las caracteristicas tienen mucha correlacion
    cuando se necesita una mejora en la eficiencia computacional
    cuando le modelo muestra señales de sobreajuste. 

## 7.7 Clustering con K-Means (No Supervisado)
El clustering es útil cuando queremos agrupar datos en categorías sin tener etiquetas.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Datos de ejemplo
X = df.drop('target', axis=1)

# K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X)

# Visualización de los clusters
plt.scatter(X['feature1'], X['feature2'], c=kmeans.labels_, cmap='viridis')
plt.title("Clusters con K-Means")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

Explicación:

KMeans() agrupa los datos en el número de clusters especificado. Los puntos dentro de cada cluster son más similares entre sí que a los puntos de otros clusters.

## 7.8 KNN (Clasificación)

In [ ]:
# Importación de bibliotecas necesarias
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris

# Cargar datos de ejemplo (Iris dataset)
data = load_iris()
X = data.data
y = data.target

# Preprocesamiento: Estándarizar las características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir el conjunto de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Crear el modelo KNN con k=3
knn = KNeighborsClassifier(n_neighbors=3)

# Entrenar el modelo
knn.fit(X_train, y_train)

# Hacer predicciones
y_pred = knn.predict(X_test)

# Evaluar el modelo
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

# 8. Entrenamiento del Modelo
- Objetivos: Ajustar los parámetros del modelo a los datos de entrenamiento.
- Acciones:
    - Entrenar con el conjunto de entrenamiento.
    - Ajustar parámetros iniciales.
    - Guardar modelos iniciales para evaluación.

# 9. Evaluación Inicial
- Objetivos: Medir el rendimiento del modelo con datos de validación.
- Acciones:
    - Clasificación: Accuracy, Precision, Recall, F1, ROC-AUC.
    - Regresión: RMSE, MAE, R².
    - Comparar varios modelos: Identificar cuál tiene el mejor desempeño.

# 10. Optimización del Modelo
- Objetivos: Mejorar el rendimiento del modelo.
- Acciones: 
    - Ajuste de hiperparámetros
        - Usar técnicas como GridSearchCV o RandomizedSearchCV.
    - Reducción de sobreajuste:
        - Regularización (L1, L2).
    - Simplificar el modelo o reducir el número de características.
    - Ensamblado: Combinar modelos para mejorar la precisión (bagging, boosting, stacking).

# 11. Evaluación Final con Datos de Prueba
- Objetivos: Verificar el desempeño en datos no vistos.
- Acciones:
    - Usar el conjunto de prueba: Obtener métricas finales de rendimiento.
    - Comparar con la línea base (baseline): Evaluar si el modelo supera métodos simples como un promedio o un modelo aleatorio.

# 12. Implementación
- Objetivos: Desplegar el modelo en un entorno real para uso práctico.
- Acciones:
    - Crear una API: Usar Flask, FastAPI o Django para servir el modelo.
    - Automatizar predicciones: Integrar con sistemas existentes.
    - Alojar el modelo: Usar servicios en la nube (AWS, Azure, GCP) o on-premises.

# 13. Monitoreo y Mantenimiento
- Objetivos: Asegurar que el modelo sigue siendo útil y preciso.
- Acciones:
    - Monitoreo de desempeño: Revisar métricas en tiempo real.
    - Identificar data drift: Detectar si los datos de entrada han cambiado.
    - Reentrenar el modelo: Actualizarlo periódicamente con nuevos datos.